##### What this notebook for:

It interacts strictly with external cloud resources and the AWS infrastructure layer.

##### Its Core Job:

It takes the secret download URL, pulls the heavy .zip archive straight into the instance's temporary memory buffer (/tmp/), extracts the raw .tsv files, uploads them safely to your Amazon S3 bucket, and then self-destructs the temporary folder completely to keep your storage bill at zero.

##### Frequency:

We only run this notebook once at the very beginning of the hackathon to safely move the data from Unstop into your cloud playground. Once your S3 bucket is populated, you never need to run it again.

In [3]:
!pwd

/home/sagemaker-user/AWS_ML2026_c_delete/AWS_ML_challengees-2026Launch-01s_/notebook


In [9]:
import os
from pathlib import Path

# Define the canonical relative pathways
current_dir = Path(os.getcwd())
base_src_path = current_dir.parent/"Package"/"code"/"business_entity_resolution"/"src"

# Create directories for source code files, pre-compiled wheels, and models
os.makedirs(os.path.join(base_src_path / "models"), exist_ok=True)
os.makedirs(os.path.join(base_src_path / "wheels"), exist_ok=True)

print(f"Success! Directories created successfully at path:\n\n -> {base_src_path} \n")
print("-> Root workspace folders successfully staged inside SageMaker Studio.")

Success! Directories created successfully at path:

 -> /home/sagemaker-user/AWS_ML2026_c_delete/AWS_ML_challengees-2026Launch-01s_/Package/code/business_entity_resolution/src 

-> Root workspace folders successfully staged inside SageMaker Studio.


(Note: S3 bucket names must be entirely unique globally).

### Run this command to create a unique bucket in your active AWS region:

_____________________________________________________________________________

#### bash
===============================================================================================

    aws s3api create-bucket \
      --bucket amazon-ml-challenge-2026-bucket \
      --region ap-north-1 \
      --create-bucket-configuration LocationConstraint=ap-north-1

  
  ---------------------------------------------------------------------------

In [17]:
import os
import zipfile
import urllib.request
import shutil
import boto3

# 1. Establish data link variables matching your team profile
challenge_data_url = '' # You need to fill it so that notebook can access data stream. It could be a link that would look like this - 'https://fg4syt3j.r.ap-north-1.awstrack.me/L0/https:%7F%7Fcdn.unstop.com%..'. We already had processed data, we have checked that the notebook architecture is working, but the dataset we processed was extremely large. So we wouldn't be providing any data in any file of the repository, the data-related files will be blank.
s3_bucket = "amazon-ml-challenge-2026-s3-bucket"   # <── Change this to your s3-bucket name

# Isolated, flat scratchpad target directory on the volatile instance
tmp_dir = "/tmp/challenge_data_flat"
zip_path = os.path.join(tmp_dir, "student_resource.zip")

os.makedirs(tmp_dir, exist_ok=True)
s3_client = boto3.client('s3')

print("-> Streaming archive directly into volatile /tmp/ scratchpad partition...")
urllib.request.urlretrieve(challenge_data_url, zip_path)

print("-> Initializing smart zip parsing extraction loop...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for member in zip_ref.infolist():
        # Target ONLY .tsv files and bypass directory index indicators
        if member.filename.endswith('.tsv') and not member.is_dir():
            # STRIP PATHS: Extract the pure filename base, ignoring internal folder structures
            filename_base = os.path.basename(member.filename)
            local_target_path = os.path.join(tmp_dir, filename_base)
            
            # Read bytes directly from the archive object and save them to a flat file layout
            with zip_ref.open(member) as source_stream, open(local_target_path, 'wb') as dest_file:
                shutil.copyfileobj(source_stream, dest_file)
            
            # Determine canonical S3 tracking prefixes dynamically based on file names
            # This fixes the open-set parsing behavior!
            if "train" in filename_base:
                s3_key = f"raw-data/train/{filename_base}"
            elif "test" in filename_base:
                s3_key = f"raw-data/test/{filename_base}"
            else:
                s3_key = f"raw-data/misc/{filename_base}"
                
            print(f"   Streaming Flat File: {filename_base} ──► s3://{s3_bucket}/{s3_key}")
            s3_client.upload_file(local_target_path, s3_bucket, s3_key)

print("-> Initiating environment self-destruct volume wipe...")
# Completely erase the staging workspace to reset instance storage utilization to zero
shutil.rmtree(tmp_dir)
print("✅ Success! Standardized flat data directories successfully synced to Amazon S3.")


-> Streaming archive directly into volatile /tmp/ scratchpad partition...
-> Initializing smart zip parsing extraction loop...
   Streaming Flat File: test_source2.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/test/test_source2.tsv
   Streaming Flat File: ._test_source2.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/test/._test_source2.tsv
   Streaming Flat File: test_source3.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/test/test_source3.tsv
   Streaming Flat File: ._test_source3.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/test/._test_source3.tsv
   Streaming Flat File: test_source1.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/test/test_source1.tsv
   Streaming Flat File: ._test_source1.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/test/._test_source1.tsv
   Streaming Flat File: train_ground_truth.tsv ──► s3://amazon-ml-challenge-2026-bucket/raw-data/train/train_ground_truth.tsv
   Streaming Flat File: ._train_ground_truth.tsv ──► s3:/

Concluding what we have done above here.

```

[Download Link] 
         │ 
         ▼ (Temporary Volatile Stream)
┌────────────────────────────────┐
│   Instance /tmp/ Scratchpad    │ <── (Wiped out instantly post-run)
└────────────────────────────────┘
         │
         ▼ (Programmatic Push via Boto3)
┌────────────────────────────────┐
│   Amazon S3 Bucket Storage     │ <── Data lives here permanently  <── We performed this much part in this notebook.
└────────────────────────────────┘
         │
         ▼ (Stream on-demand via Fast File Mode)
┌────────────────────────────────┐
│ SageMaker JupyterLab Space     │ <── Notebook disk stays empty
└────────────────────────────────┘

```

The .tsv data files are created directly inside our secure Amazon S3 bucket under the unified flat prefix path amazon-ml-challenge-2026-bucket.
They are not taking up space inside visible notebook directory (AWS_submission_2026), which means persistent workspace disk storage stays completely empty.
Here is the exact path layout where the execution loop has safely placed our files inside the AWS ecosystem:

``` text

s3://amazon-ml-challenge-2026-bucket/
└── raw-data/
    ├── train/
    │   ├── train_source1.tsv      <── Reference Ground Truth
    │   ├── train_source2.tsv      <── Target Source 2 Train Data
    │   ├── train_source3.tsv      <── Target Source 3 Train Data
    │   └── train_ground_truth.tsv <── Target Matching Labels
    └── test/
        ├── test_source1.tsv       <── Evaluated Test Set Entities
        ├── test_source2.tsv       <── Target Source 2 Inference Data
        └── test_source3.tsv       <── Target Source 3 Inference Data

```
How Do We Read Them from the Notebook Now?

Now that the data is securely hosted inside the S3 storage cloud, you will use SageMaker Fast File Mode to interact with them inside your main workspace.When you boot up data_ingestion.ipynb, SageMaker mounts these exact S3 paths as a virtual local directory (e.g., /opt/ml/processing/input/). Your python scripts stream row entries directly out of S3 straight into RAM memory buffers on demand, protecting your notebook instance kernel from memory spikes and crash timeouts.

To completely delete your challenge bucket and data from the terminal instantly:

```bash

# First, empty all data inside the bucket safely
aws s3 rm s3://amazon-ml-challenge-2026-bucket --recursive

# Then, delete the bucket infrastructure allocation
aws s3 rb s3://amazon-ml-challenge-2026-bucket

```